In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, TimestampType, DateType
import pyspark.sql.functions as f
catalog_name= "ecommerce" 


**Brand Silver Table**

In [0]:
df_bronze = spark.table(f"{catalog_name}.bronze.brz_brands")
df_silver= df_bronze.withColumn("brand_name",f.trim(f.col("brand_name")))
df_silver = df_silver.withColumn("brand_code",f.regexp_replace(f.col("brand_code"), r"[^a-zA-Z0-9]", ""))

In [0]:
anomalies={
    "BOOKS" : "BKS",
    "GROCERY" : "GRCY",
    "TOYS" : "TOY"
}
df_silver= df_silver.replace(anomalies,subset=["category_code"])
df_silver.write.mode("overwrite")\
    .option("mergeSchema", "true")\
    .saveAsTable(f"{catalog_name}.silver.slv_brands")

**Category Silver Table**

In [0]:
df_bronze = spark.table(f"{catalog_name}.bronze.brz_category")
# df_duplicate= df_bronze.groupBy("category_code").count().filter(f.col("count")>1)
df_silver= df_bronze.dropDuplicates(["category_code"])
df_silver = df_silver.withColumn("category_code",f.upper(f.col("category_code")))
df_silver.write.mode("overwrite")\
    .option("mergeSchema", "true")\
    .saveAsTable(f"{catalog_name}.silver.slv_category") 

**Product Silver Table**

In [0]:
df_bronze = spark.table(f"{catalog_name}.bronze.brz_product")
df_silver= df_bronze.withColumn("weight_grams",f.regexp_replace(f.col("weight_grams"), "g", "").cast(IntegerType()))\
    .withColumn("length_cm",f.regexp_replace(f.col("length_cm"), ",", ".").cast(FloatType()))\
    .withColumn("brand_code",f.upper(f.col("brand_code")))\
    .withColumn("category_code",f.upper(f.col("category_code")))\
    .withColumn("material",
                f.when(f.col("material")=='Coton','Cotton')
                 .when(f.col("material")=='Alumium','Aluminum')
                 .when(f.col("material")=='Ruber','Rubber')
                 .otherwise(f.col("material")))\
    .withColumn("rating_count",
                f.when(f.col("rating_count").isNotNull(),f.abs(f.col("rating_count")))
                 .otherwise(f.col("rating_count"))) 

df_silver.write.mode("overwrite")\
    .option("mergeSchema", "true")\
    .saveAsTable(f"{catalog_name}.silver.slv_product") 

**Customer Silver Table**

In [0]:
df_bronze = spark.table(f"{catalog_name}.bronze.brz_customer")
df_silver= df_bronze.dropna(subset=["customer_id"])\
    .withColumn("phone", f.when(f.col("phone").isNull(), "Not Available")
                .otherwise(f.when(f.col("phone") != "Not Available",
                                  f.concat(f.lit("+"), f.regexp_replace(f.col("phone"), r"\.0$", "")))
                           .otherwise(f.col("phone"))))
df_silver.write.mode("overwrite")\
    .option("mergeSchema", "true")\
    .saveAsTable(f"{catalog_name}.silver.slv_customer") 


**Date Silver Table**

In [0]:
df_bronze= spark.table(f"{catalog_name}.bronze.brz_date")
df_silver= df_bronze.withColumn("date",f.to_date(f.col("date"),"dd-MM-yyyy"))\
                    .dropDuplicates(["date"])\
                    .withColumn("day_name",f.initcap(f.col("day_name")))\
                    .withColumn("week_of_year",f.concat_ws("-", f.concat(f.lit("Week"), f.abs(f.col("week_of_year")), f.lit("-"), f.col("year"))))\
                    .withColumn("quarter",f.concat_ws("-", f.concat(f.lit("Q"), f.abs(f.col("quarter")), f.lit("-"), f.col("year"))))\
                    .withColumnRenamed("week_of_year","week")

df_silver.write.mode("overwrite")\
    .option("mergeSchema", "true")\
    .saveAsTable(f"{catalog_name}.silver.slv_calendar")